# 🏭 Industrial Surface Defect Detection — Demo Notebook

**Course:** AIC2022 — Artificial Intelligence Course  
**Topic:** Computer Vision · Transfer Learning · Binary Image Classification

---

## Overview
This notebook walks through the complete surface defect detection pipeline:

| Section | Topic |
|---------|-------|
| 1 | Introduction & Environment Setup |
| 2 | Data Exploration & Visualisation |
| 3 | Model Architecture (ResNet18 Transfer Learning) |
| 4 | Training |
| 5 | Evaluation & Metrics |
| 6 | Inference Demo |

> **Dataset**: [Kaggle Casting Product Defect Dataset](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product)

---
## Section 1 — Introduction & Setup

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import sys
import random
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import src.*
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Third-party ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.datasets import ImageFolder

# ── Project modules ─────────────────────────────────────────────────────────
from src.config import (
    DEVICE, IMAGE_SIZE, BATCH_SIZE, EPOCHS, LEARNING_RATE,
    CLASS_NAMES, DATA_DIR, MODEL_SAVE_PATH, RESULTS_DIR, SEED
)
from src.utils import set_seed, get_transforms, compute_metrics, plot_training_history
from src.model import get_model, EarlyStopping

set_seed(SEED)

print(f'PyTorch version : {torch.__version__}')
print(f'Torchvision     : {torchvision.__version__}')
print(f'Device          : {DEVICE}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

---
## Section 2 — Data Exploration & Visualisation

We explore the dataset structure, class distribution, and visualise sample images.

In [ ]:
# ── Dataset paths ────────────────────────────────────────────────────────────
TRAIN_DIR = os.path.join(PROJECT_ROOT, 'data', 'train')
TEST_DIR  = os.path.join(PROJECT_ROOT, 'data', 'test')

assert os.path.isdir(TRAIN_DIR), f'Training directory not found: {TRAIN_DIR}'

# Count images per class
print('Training set class counts:')
class_counts = {}
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([
            f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ])
        class_counts[cls] = count
        print(f'  {cls:10s}: {count} images')

total = sum(class_counts.values())
print(f'\n  Total: {total} training images')

In [ ]:
# ── Class distribution bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
ax = axes[0]
colors = ['#e74c3c', '#2ecc71']
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black', linewidth=0.8)
ax.set_title('Class Distribution (Training Set)', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Number of Images')
for bar, (cls, cnt) in zip(bars, class_counts.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(cnt), ha='center', fontsize=11, fontweight='bold')

# Pie chart
ax = axes[1]
ax.pie(class_counts.values(), labels=class_counts.keys(),
       autopct='%1.1f%%', colors=colors, startangle=90,
       textprops={'fontsize': 11})
ax.set_title('Class Balance', fontsize=13)

plt.suptitle('Dataset Overview', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise sample images ──────────────────────────────────────────────────
from PIL import Image

def show_samples(data_dir, classes, n_per_class=4, figsize=(14, 6)):
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=figsize)
    for row, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        images  = [
            f for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ][:n_per_class]
        for col, img_name in enumerate(images):
            img = Image.open(os.path.join(cls_dir, img_name)).convert('RGB')
            ax  = axes[row][col]
            ax.imshow(img)
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(cls.upper(), fontsize=12, fontweight='bold',
                              rotation=0, labelpad=50, va='center')
    plt.suptitle('Sample Images — Training Set', fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples(TRAIN_DIR, sorted(class_counts.keys()))

In [ ]:
# ── Visualise data augmentation ──────────────────────────────────────────────
import torchvision.transforms.functional as TF

# Pick one sample image
sample_cls = sorted(class_counts.keys())[0]
sample_dir = os.path.join(TRAIN_DIR, sample_cls)
sample_img_path = os.path.join(sample_dir, os.listdir(sample_dir)[0])
original = Image.open(sample_img_path).convert('RGB')

train_transform = get_transforms('train')

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes[0][0].imshow(original)
axes[0][0].set_title('Original', fontsize=10)
axes[0][0].axis('off')

mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])

for i in range(1, 10):
    row, col = divmod(i, 5)
    aug = train_transform(original)
    # Denormalise for display
    aug_display = aug * std[:, None, None] + mean[:, None, None]
    aug_display = aug_display.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[row][col].imshow(aug_display)
    axes[row][col].set_title(f'Augmented {i}', fontsize=9)
    axes[row][col].axis('off')

plt.suptitle('Training Data Augmentation Examples', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 3 — Model Architecture

We use **ResNet18** pretrained on ImageNet. The strategy:

1. Load pretrained weights → features already useful for general image recognition
2. **Freeze** all layers except `layer4` and the final `fc`
3. Replace `fc` with `Linear(512 → 2)` for binary classification

This greatly reduces training time while leveraging powerful low-level features.

In [ ]:
# ── Inspect model ────────────────────────────────────────────────────────────
model = get_model(num_classes=2, pretrained=True)

print('Layer-wise trainability:')
print(f'{"Layer":<30} {"Trainable":<12} {"# Params":>12}')
print('-' * 56)
for name, module in model.named_children():
    params     = sum(p.numel() for p in module.parameters())
    trainable  = sum(p.numel() for p in module.parameters() if p.requires_grad)
    status     = '✓ YES' if trainable > 0 else '✗ frozen'
    print(f'{name:<30} {status:<12} {params:>12,}')

total_p    = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('-' * 56)
print(f'  Total params     : {total_p:,}')
print(f'  Trainable params : {trainable_p:,}  ({100*trainable_p/total_p:.1f}%)')

---
## Section 4 — Training

We run training directly from this notebook. Alternatively call the CLI:
```bash
python src/train.py --data_dir data/train --epochs 10
```

In [ ]:
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm

# ── Hyperparameters ──────────────────────────────────────────────────────────
NUM_EPOCHS  = 10       # reduce to 2–3 for a quick test
LR          = 0.001
BSIZE       = 32

# ── Dataset ──────────────────────────────────────────────────────────────────
full_ds     = ImageFolder(root=TRAIN_DIR)
n_total     = len(full_ds)
n_train     = int(0.85 * n_total)
n_val       = n_total - n_train

train_sub, val_sub = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply transforms through wrapper
class TransformDataset(torch.utils.data.Dataset):
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):  return len(self.base)
    def __getitem__(self, idx):
        img, lbl = self.base.dataset[self.base.indices[idx]]
        return self.transform(img), lbl

train_ds = TransformDataset(train_sub, get_transforms('train'))
val_ds   = TransformDataset(val_sub,   get_transforms('val'))

train_loader = DataLoader(train_ds, batch_size=BSIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BSIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} images  |  Val: {len(val_ds)} images')
print(f'Class names: {full_ds.classes}')

In [ ]:
# ── Initialise model, loss, optimiser ────────────────────────────────────────
model     = get_model(num_classes=2, pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = StepLR(optimizer, step_size=3, gamma=0.1)
early_stop = EarlyStopping(patience=3)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [train]', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        _, preds  = torch.max(out, 1)
        t_loss   += loss.item() * imgs.size(0)
        t_correct += (preds == lbls).sum().item()
        t_total   += imgs.size(0)

    # --- Validate ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [val]  ', leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, lbls)
            _, preds  = torch.max(out, 1)
            v_loss   += loss.item() * imgs.size(0)
            v_correct += (preds == lbls).sum().item()
            v_total   += imgs.size(0)

    t_avg_loss = t_loss / t_total;  t_acc = t_correct / t_total
    v_avg_loss = v_loss / v_total;  v_acc = v_correct / v_total

    train_losses.append(t_avg_loss);  val_losses.append(v_avg_loss)
    train_accs.append(t_acc);         val_accs.append(v_acc)

    print(f'Epoch {epoch:>2}/{NUM_EPOCHS} '
          f'| Train loss: {t_avg_loss:.4f}  acc: {t_acc*100:.2f}% '
          f'| Val loss: {v_avg_loss:.4f}  acc: {v_acc*100:.2f}%')

    # Save best
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        os.makedirs(os.path.join(PROJECT_ROOT, 'models'), exist_ok=True)
        torch.save({
            'epoch': epoch, 'val_acc': v_acc,
            'model_state_dict': model.state_dict(),
            'class_names': full_ds.classes,
        }, MODEL_SAVE_PATH)
        print(f'  → Best model saved (val_acc={v_acc*100:.2f}%)')

    scheduler.step()
    early_stop(v_avg_loss)
    if early_stop.stop:
        print('Early stopping.')
        break

print(f'\nBest validation accuracy: {best_val_acc*100:.2f}%')

In [ ]:
# ── Plot training history ────────────────────────────────────────────────────
curves_path = os.path.join(PROJECT_ROOT, RESULTS_DIR, 'training_curves.png')
plot_training_history(train_losses, val_losses, train_accs, val_accs, curves_path)

# Display inline
from IPython.display import Image as IPyImage
IPyImage(curves_path)

---
## Section 5 — Evaluation & Metrics

We evaluate the best saved model on the held-out test set.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torchvision.datasets import ImageFolder

# ── Load best model ──────────────────────────────────────────────────────────
checkpoint   = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
eval_model   = get_model(num_classes=2, pretrained=False).to(DEVICE)
eval_model.load_state_dict(checkpoint['model_state_dict'])
eval_model.eval()
class_names  = checkpoint.get('class_names', CLASS_NAMES)
print(f'Loaded model from epoch {checkpoint["epoch"]}  val_acc={checkpoint["val_acc"]*100:.2f}%')

# ── Test dataset ─────────────────────────────────────────────────────────────
test_ds = ImageFolder(root=TEST_DIR, transform=get_transforms('test'))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
print(f'Test samples: {len(test_ds)}')

# ── Run inference ────────────────────────────────────────────────────────────
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Testing'):
        imgs = imgs.to(DEVICE)
        _, preds = torch.max(eval_model(imgs), 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.numpy())

# ── Metrics ──────────────────────────────────────────────────────────────────
metrics = compute_metrics(all_labels, all_preds)
print('\n' + '='*50)
print('         TEST SET RESULTS')
print('='*50)
for k, v in metrics.items():
    print(f'  {k.capitalize():<12}: {v*100:.2f}%' if k == 'accuracy' else f'  {k.capitalize():<12}: {v:.4f}')
print('='*50)

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)
plt.tight_layout()

cm_path = os.path.join(PROJECT_ROOT, RESULTS_DIR, 'confusion_matrix.png')
os.makedirs(os.path.dirname(cm_path), exist_ok=True)
fig.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {cm_path}')

In [ ]:
# ── Full classification report ────────────────────────────────────────────────
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

---
## Section 6 — Inference Demo

Run the trained model on individual images and visualise predictions.

In [ ]:
import torch.nn.functional as F

def predict_pil(pil_image, model, class_names, device=DEVICE):
    """Predict class and confidence for a PIL image."""
    transform = get_transforms('test')
    tensor    = transform(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).squeeze(0)
    pred_idx   = probs.argmax().item()
    return class_names[pred_idx].upper(), float(probs[pred_idx]), probs.cpu().numpy()


def show_predictions(data_dir, class_names, model, n=8, device=DEVICE):
    """Show n random test images with their predictions."""
    all_imgs = []
    for cls in class_names:
        cls_dir = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = [
            (os.path.join(cls_dir, f), cls)
            for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        all_imgs.extend(imgs)

    sample = random.sample(all_imgs, min(n, len(all_imgs)))
    fig, axes = plt.subplots(2, n // 2, figsize=(16, 7))
    axes = axes.ravel()

    for i, (img_path, true_cls) in enumerate(sample):
        pil_img = Image.open(img_path).convert('RGB')
        pred_label, confidence, probs = predict_pil(pil_img, model, class_names, device)
        is_correct = pred_label.lower() == true_cls.lower()

        axes[i].imshow(pil_img)
        color = '#2ecc71' if is_correct else '#e74c3c'
        icon  = '✓' if is_correct else '✗'
        axes[i].set_title(
            f'{icon} Pred: {pred_label}\n{confidence*100:.1f}%  (GT: {true_cls.upper()})',
            color=color, fontsize=9, fontweight='bold'
        )
        axes[i].axis('off')

    plt.suptitle('Random Test Set Predictions', fontsize=14)
    plt.tight_layout()
    plt.show()


show_predictions(TEST_DIR, class_names, eval_model)

In [ ]:
# ── Predict a single image by path ──────────────────────────────────────────
# Change this path to any image you want to test:
image_path = os.path.join(TEST_DIR, class_names[0], os.listdir(os.path.join(TEST_DIR, class_names[0]))[0])

pil_img = Image.open(image_path).convert('RGB')
label, conf, probs_arr = predict_pil(pil_img, eval_model, class_names)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(pil_img)
axes[0].set_title(f'Input Image', fontsize=12)
axes[0].axis('off')

colors = ['#e74c3c' if c == 'defect' else '#2ecc71' for c in class_names]
bars   = axes[1].barh(class_names, probs_arr * 100, color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Confidence (%)', fontsize=11)
axes[1].set_title(f'Prediction: {label}  ({conf*100:.1f}%)', fontsize=12, fontweight='bold')
for bar, p in zip(bars, probs_arr):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{p*100:.1f}%', va='center', fontsize=10)

plt.suptitle('Single Image Inference', fontsize=13)
plt.tight_layout()
plt.show()